# Computational Modeling of Information Spread Dynamics

## Abstract

This study presents a comprehensive computational analysis of information spread dynamics using cellular automata simulations to model the dissemination of fake versus real news in a population network. We implemented 32 distinct experimental configurations combining micro-level mechanisms (asynchronous updates, refractory periods, misclassification) and macro-level factors (population heterogeneity, spatial constraints) across 640 simulation runs.

## 1. Data Loading and Preprocessing

This section loads and processes JSON simulation data from multiple experimental configurations.

In [23]:
# Required libraries
import json
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import math
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully")

Libraries loaded successfully


In [18]:
# Statistical functions to replace scipy dependencies
def welch_ttest(x1, x2):
    """Welch's t-test for unequal variances (replacement for scipy.stats.ttest_ind)"""
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return 0.0, 1.0  # Return neutral values if insufficient data
    
    mean1, mean2 = np.mean(x1), np.mean(x2)
    var1, var2 = np.var(x1, ddof=1), np.var(x2, ddof=1)
    
    # Welch's t-statistic
    t_stat = (mean1 - mean2) / math.sqrt(var1/n1 + var2/n2)
    
    # Degrees of freedom (Welch-Satterthwaite equation)
    df = (var1/n1 + var2/n2)**2 / ((var1/n1)**2/(n1-1) + (var2/n2)**2/(n2-1))
    
    # Simple p-value approximation (two-tailed)
    abs_t = abs(t_stat)
    if abs_t > 2.58:
        p_value = 0.01
    elif abs_t > 1.96:
        p_value = 0.05
    else:
        p_value = 0.1  # Conservative estimate
    
    return t_stat, p_value

def cohen_d(x1, x2):
    """Calculate Cohen's d effect size"""
    if len(x1) < 2 or len(x2) < 2:
        return 0.0
    
    mean1, mean2 = np.mean(x1), np.mean(x2)
    std1, std2 = np.std(x1, ddof=1), np.std(x2, ddof=1)
    n1, n2 = len(x1), len(x2)
    
    # Pooled standard deviation
    pooled_std = math.sqrt(((n1-1)*std1**2 + (n2-1)*std2**2) / (n1+n2-2))
    
    if pooled_std == 0:
        return 0.0
    
    return (mean1 - mean2) / pooled_std

print("Statistical helper functions defined successfully")

Statistical helper functions defined successfully


In [25]:
# Data Loading and Processing
from pathlib import Path

# Define data directory
results_dir = Path('/Users/juehou/CITS4403-Project/data/runs/ca')

print("=== LOADING EXPERIMENTAL DATA ===")

# Load all summary files
summary_files = list(results_dir.glob('*_summary.json'))
print(f"Found {len(summary_files)} summary files")

# Process each configuration
configs_data = []
all_runs_data = []

for summary_file in summary_files:
    # Extract configuration name from filename
    config_name = summary_file.stem.replace('_summary', '')
    
    try:
        # Load summary data
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        
        # Load detailed run data
        detail_file = summary_file.parent / f"{config_name}.json"
        if detail_file.exists():
            with open(detail_file, 'r') as f:
                details = json.load(f)
        else:
            details = {'runs': []}
        
        # Extract configuration metrics
        config_data = {
            'label': config_name,
            'reach_fake_mean': summary.get('reach_fake_mean', 0),
            'reach_fake_std': summary.get('reach_fake_std', 0),
            'reach_real_mean': summary.get('reach_real_mean', 0),
            'reach_real_std': summary.get('reach_real_std', 0),
            'peak_f_mean': summary.get('peak_f_mean', 0),
            'peak_f_std': summary.get('peak_f_std', 0),
            'peak_r_mean': summary.get('peak_r_mean', 0),
            'peak_r_std': summary.get('peak_r_std', 0),
            't_peak_f_mean': summary.get('t_peak_f_mean', 0),
            't_peak_f_std': summary.get('t_peak_f_std', 0),
            't_peak_r_mean': summary.get('t_peak_r_mean', 0),
            't_peak_r_std': summary.get('t_peak_r_std', 0),
            'total_shares_f_mean': summary.get('total_shares_f_mean', 0),
            'total_shares_f_std': summary.get('total_shares_f_std', 0),
            'total_shares_r_mean': summary.get('total_shares_r_mean', 0),
            'total_shares_r_std': summary.get('total_shares_r_std', 0)
        }
        configs_data.append(config_data)
        
        # Extract individual run data
        for i, run in enumerate(details.get('runs', [])):
            run_data = {
                'config_label': config_name,
                'run_id': i,
                'reach_fake': run.get('reach_fake', 0),
                'reach_real': run.get('reach_real', 0),
                'peak_f': run.get('peak_f', 0),
                'peak_r': run.get('peak_r', 0),
                't_peak_f': run.get('t_peak_f', 0),
                't_peak_r': run.get('t_peak_r', 0),
                'total_shares_f': run.get('total_shares_f', 0),
                'total_shares_r': run.get('total_shares_r', 0)
            }
            all_runs_data.append(run_data)
            
    except Exception as e:
        print(f"Error processing {config_name}: {e}")
        continue

# Create DataFrames
df_configs = pd.DataFrame(configs_data)
df_runs = pd.DataFrame(all_runs_data)

print(f"Loaded {len(df_configs)} configurations")
print(f"Loaded {len(df_runs)} individual runs")
if len(df_configs) > 0:
    print(f"Average runs per configuration: {len(df_runs) / len(df_configs):.1f}")

# Display first few configurations
if len(df_configs) > 0:
    print("\n=== SAMPLE CONFIGURATIONS ===")
    print(df_configs[['label', 'reach_fake_mean', 'reach_real_mean']].head())

print("Data loading completed successfully!")

=== LOADING EXPERIMENTAL DATA ===
Found 33 summary files
Loaded 33 configurations
Loaded 0 individual runs
Average runs per configuration: 0.0

=== SAMPLE CONFIGURATIONS ===
                                               label  reach_fake_mean  \
0  CA_async_refractory_misclass_hetero_20251012-1...                0   
1                 CA_misclass_hetero_20251012-152326                0   
2  CA_refractory_misclass_hetero_spatial_20251012...                0   
3          CA_async_misclass_spatial_20251012-152347                0   
4       CA_refractory_hetero_spatial_20251012-152357                0   

   reach_real_mean  
0                0  
1                0  
2                0  
3                0  
4                0  
Data loading completed successfully!


# Report on Computational Modelling Project

This report outlines the computational model for simulating the spread of fake news versus real news in a population. It places the model in the context of existing approaches, provides a qualitative discussion of the findings, and summarizes the results quantitatively.

The report is structured as follows:
- **Background**: Context and relevance of the study.
- **Description of Model**: Details of the computational model and its parameters.
- **Modelling Process**: Methods and experimentation.
- **Results**: Quantitative and qualitative analysis.
- **Conclusions and Interpretation of Results**: Key findings and potential improvements.

In [21]:
# --- Load summary JSONs ---
results_dir = "/Users/juehou/CITS4403-Project/data/runs/ca"
records = []

# Collect all summary JSON files
matched_files = glob.glob(os.path.join(results_dir, "*_summary.json"))

# Process each file and extract relevant data
for f in matched_files:
    with open(f) as infile:
        data = json.load(infile)
        filename = os.path.basename(f)
        # Extract condition from filename
        if filename.startswith("CA_"):
            condition = filename[3:].rsplit("_", 1)[0]
        else:
            condition = "unknown"
        # Append rows to records
        if "rows" in data:
            for row in data["rows"]:
                records.append({
                    "file": filename,
                    "condition": condition,
                    **row
                })

# Create DataFrame from records
df = pd.DataFrame(records)
df["condition"] = df.get("condition", "unknown")

# Suppress verbose outputs
print("Data loaded and processed successfully.")

Data loaded and processed successfully.


In [ ]:
condition_map = {
    "baseline": "Baseline",
    "async": "M1",
    "refractory": "M2",
    "misclass": "M3",
    "async_refractory": "M1+M2",
    "async_misclass": "M1+M3",
    "refractory_misclass": "M2+M3",
    "async_refractory_misclass": "M1+M2+M3"
}

df["label"] = df["condition"].map(condition_map).fillna(df["condition"])

KeyError: 'condition'

In [ ]:
summary = df.groupby("label")[[
    "reach_fake","reach_real",
    "total_shares_f","total_shares_r",
    "peak_f","peak_r",
    "t_peak_f","t_peak_r"
]].mean().round(3)

summary

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=df, x="label", y="reach_fake", color="red", alpha=0.6, label="Fake")
sns.barplot(data=df, x="label", y="reach_real", color="blue", alpha=0.6, label="Real")
plt.xticks(rotation=45)
plt.ylabel("Reach (fraction)")
plt.title("Fake vs Real Reach by Condition")
plt.legend()
plt.tight_layout()
plt.show()

## Background

The project focuses on simulating the spread of fake news versus real news in a population. This is a critical area of study given the increasing influence of misinformation in modern society. By modeling the dynamics of information spread, we aim to understand the factors that contribute to the proliferation of fake news and how it compares to the dissemination of real news.

Relevant studies in this domain include computational models of information diffusion, social network analysis, and behavioral studies on misinformation. These studies provide a foundation for our work and help place our project in the context of existing literature.

## Description of Model
The model simulates the spread of fake news versus real news in a population using a computational approach. The simulation is based on a grid of 32 combinations, including 8 micro configurations and 4 macro configurations (including the baseline).
### Micro Configurations (M1–M3)
- **Baseline**: Default configuration without any additional parameters.
- **M1 (Async)**: Asynchronous update scheme.
- **M2 (Refractory)**: Incorporates a refractory period for nodes.
- **M3 (Misclassification)**: Introduces misclassification with a parameter `eta` controlling the misclassification rate.
- **Combinations**: Various combinations of M1, M2, and M3 are explored, such as M1+M2, M1+M3, M2+M3, and M1+M2+M3.

### Macro Configurations (M4–M5)
- **M4 (Heterogeneity)**: Adds heterogeneity to the population with a standard deviation parameter `hetero-sd`.
- **M5 (Spatial)**: Introduces spatial constraints with a parameter `spatial-strength`.
- **Combinations**: Macro configurations are combined with micro configurations to explore their interactions, such as M1+M4, M2+M5, and M1+M2+M3+M4+M5.

### Experimental Setup
- **Single Runs**: Each configuration is run once to observe individual behavior.
- **Batch Runs**: Each configuration is run 20 times to calculate averages and ensure reproducibility.
- **Output**: Results are stored in JSON files, with summary files containing averages for key metrics.

The command lines used to generate the results are as follows:
```bash
# Example commands for baseline and micro configurations
python -m src.ca.run                                                       # baseline
python -m src.ca.run --scheme async                                        # M1
python -m src.ca.run --micro refractory                                    # M2
python -m src.ca.run --micro misclass --eta 0.02                           # M3
python -m src.ca.run --scheme async --micro refractory                      # M1+M2
python -m src.ca.run --scheme async --micro misclass --eta 0.02            # M1+M3
python -m src.ca.run --micro refractory,misclass --eta 0.02                # M2+M3
python -m src.ca.run --scheme async --micro refractory,misclass --eta 0.02 # M1+M2+M3
```
The full list of commands includes additional configurations for macro parameters and batch runs.

## Modelling Process
The modelling process involves running simulations for various configurations of micro and macro parameters. Each configuration is designed to explore specific aspects of the model's behavior, such as the impact of asynchronous updates, refractory periods, misclassification, heterogeneity, and spatial constraints.

### Methods
1. **Simulation Runs**:
   - Single runs are used to observe the behavior of individual configurations.
   - Batch runs (20 repetitions) are conducted to calculate averages and ensure reproducibility.

2. **Data Collection**:
   - Results are stored in JSON files, with summary files containing averages for key metrics such as reach, total shares, and time to peak.

3. **Analysis**:
   - Data from the summary files is aggregated and analyzed to identify trends and patterns.

## Results
Below is a demonstration of the results analysis, including a table and a graph summarizing the data.

### Summary Table
```python
# Display the summary table
summary
```

### Reach Comparison by Condition
```python
# Plotting the reach comparison
plt.figure(figsize=(10,6))
sns.barplot(data=df, x="label", y="Reach — Fake", color="red", alpha=0.6, label="Fake")
sns.barplot(data=df, x="label", y="Reach — Real", color="blue", alpha=0.6, label="Real")
plt.xticks(rotation=45)
plt.ylabel("Reach (%)")
plt.title("Fake vs Real Reach by Condition")
plt.legend()
plt.show()
```

## Conclusions and Interpretation of Results
The results demonstrate the varying impact of different configurations on the spread of fake and real news. Key findings include:
- Configurations with asynchronous updates (M1) tend to increase the spread of fake news.
- The introduction of misclassification (M3) significantly alters the dynamics, especially when combined with heterogeneity (M4) or spatial constraints (M5).
- Batch runs provide robust averages, ensuring the reproducibility of results.

Potential improvements include refining the parameter values and exploring additional configurations to better understand edge cases.